In [1]:
!pip install onnx onnxruntime onnxruntime-tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.2 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time
import psutil
import os

from torchvision.models import resnet50

# Load base model
model = resnet50(weights=None)

# 🔥 Replace final layer (VERY IMPORTANT)
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 128),
    nn.ReLU(),
    nn.Linear(128, 4)
)

# Load your trained weights
model.load_state_dict(torch.load("RESNETmodel_fp32.pth"))
model.eval()


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [3]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 12.0 MB/s eta 0:00:00


In [4]:



dummy_input = torch.randn(1, 3, 110, 100)

torch.onnx.export(
    model,
    dummy_input,
    "resnetmodel_fp32.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=11,
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)

print("ONNX model saved!")

/tmp/ipykernel_3661/4151969798.py:3: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0419 14:39:45.199000 3661 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0419 14:39:46.424000 3661 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, al

[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:68: adapt: Asserti

Applied 107 of general pattern rewrite rules.
ONNX model saved!


In [5]:
!pip install onnxsim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 8.9 MB/s eta 0:00:00


In [6]:
from onnxsim import simplify
import onnx

model = onnx.load("resnetmodel_fp32.onnx")
model_simp, check = simplify(model)

onnx.save(model_simp, "model_fp32_simplified.onnx")

In [7]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="model_fp32_simplified.onnx",
    model_output="resnetmodel_int8.onnx",
    weight_type=QuantType.QInt8
)

print("✅ INT8 ONNX model saved")

✅ INT8 ONNX model saved


Loading the test model

In [8]:
pip install gdown

In [9]:
# Download the zip file from Google Drive
import gdown
import os

# link: https://drive.google.com/file/d/1xNDwcGyhqY_7tdVc58YSJCS4DptDoEUL/view?usp=sharing
file_id = '1xNDwcGyhqY_7tdVc58YSJCS4DptDoEUL'
output_filename = 'archive.zip'

gdown.download(id=file_id, output=output_filename, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1xNDwcGyhqY_7tdVc58YSJCS4DptDoEUL
From (redirected): https://drive.google.com/uc?id=1xNDwcGyhqY_7tdVc58YSJCS4DptDoEUL&confirm=t&uuid=3821f60f-92c4-45d8-9888-3cc382b9276f
To: /content/archive.zip
100%|██████████| 1.27G/1.27G [00:31<00:00, 40.7MB/s]


'archive.zip'

In [10]:
# Extract the contents of the zip file
import zipfile

if os.path.exists(output_filename):
    with zipfile.ZipFile(output_filename, 'r') as zip_ref:
        zip_ref.extractall('.') # Extract to the current directory
    print(f"Successfully extracted '{output_filename}'")
    # Optionally, remove the zip file after extraction
    # os.remove(output_filename)
else:
    print(f"Error: The file '{output_filename}' was not found.")

Successfully extracted 'archive.zip'


In [11]:
# List the contents of the current directory to see extracted files
print("Contents of the current directory after extraction:")
print(os.listdir('.'))

Contents of the current directory after extraction:
['.config', 'resnetmodel_fp32.onnx.data', 'model_fp32_simplified.onnx', 'resnetmodel_int8.onnx', 'test', 'RESNETmodel_fp32.pth', 'resnetmodel_fp32.onnx', 'archive.zip', 'train', 'sample_data']


In [12]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

transform=transforms.Compose([
    transforms.Resize((110, 100)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

full_dataset = datasets.ImageFolder(root='train', transform=transform)

In [16]:
test_dataset = datasets.ImageFolder(root='test', transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [17]:
import onnxruntime
import numpy as np
from tqdm import tqdm

# Create an ONNX Runtime session
# Since it's an INT8 model, we can try to use CPUExecutionProvider for simplicity
sess_options = onnxruntime.SessionOptions()
# Optional: Set graph optimization level to enable more optimizations
sess_options.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_EXTENDED

sess = onnxruntime.InferenceSession("resnetmodel_int8.onnx", sess_options=sess_options, providers=['CPUExecutionProvider'])

# Get input and output names
input_name = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name

correct = 0
total = 0

print("Starting evaluation of the INT8 ONNX model...")

# Iterate over the test dataset
for inputs, labels in tqdm(test_loader, desc="Evaluating model"):
    # ONNX Runtime expects numpy arrays, so convert PyTorch tensors
    inputs_np = inputs.numpy()

    # Run inference
    outputs = sess.run([output_name], {input_name: inputs_np})

    # The output is a list, get the first (and usually only) element
    predictions = outputs[0]

    # Get predicted class (index of the max log-probability)
    predicted_labels = np.argmax(predictions, axis=1)

    total += labels.size(0)
    correct += (predicted_labels == labels.numpy()).sum().item()

accuracy = 100 * correct / total
print(f"\nAccuracy of the INT8 ONNX model on the test dataset: {accuracy:.2f}%")

Starting evaluation of the INT8 ONNX model...


Evaluating model: 100%|██████████| 72/72 [13:52<00:00, 11.57s/it]


Accuracy of the INT8 ONNX model on the test dataset: 21.19%


Memory Usage measurements

In [22]:
import onnxruntime as ort

In [23]:
import psutil, os

def get_ram():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024**2  # Convert bytes → MB

In [24]:
def measure_onnx_memory(model_path):
    print(f"\n===== Measuring {model_path} =====")

    # 1. Baseline
    baseline_ram = get_ram()
    print(f"Baseline RAM: {baseline_ram:.2f} MB")

    # 2. Load model
    session = ort.InferenceSession(
        model_path,
        providers=['CPUExecutionProvider']
    )
    input_name = session.get_inputs()[0].name

    after_load_ram = get_ram()
    print(f"After Load RAM: {after_load_ram:.2f} MB")

    # 3. Warmup
    dummy = np.random.randn(1, 3, 110, 100).astype(np.float32)
    for _ in range(10):
        _ = session.run(None, {input_name: dummy})

    # 4. Steady-state inference
    ram_before_inf = get_ram()

    for _ in range(100):
        _ = session.run(None, {input_name: dummy})

    ram_after_inf = get_ram()
    steady_ram = ram_after_inf

    print(f"Steady-State RAM: {steady_ram:.2f} MB")

    return {
        "baseline": baseline_ram,
        "after_load": after_load_ram,
        "steady": steady_ram
    }

In [25]:
fp32_results = measure_onnx_memory("model_fp32_simplified.onnx")
# int8_results = measure_onnx_memory("resnetmodel_int8.onnx")


===== Measuring model_fp32_simplified.onnx =====
Baseline RAM: 1451.73 MB
After Load RAM: 1569.61 MB
Steady-State RAM: 1569.82 MB

===== Measuring resnetmodel_int8.onnx =====
Baseline RAM: 1569.82 MB
After Load RAM: 1569.82 MB
Steady-State RAM: 1569.82 MB


In [27]:
fp32_ram = fp32_results["steady"]
int8_ram = int8_results["steady"]

reduction = ((fp32_ram - int8_ram) / fp32_ram) * 100

print(f"\nMemory Reduction: {reduction:.10f}%")


Memory Reduction: 0.0000000000%
